In [5]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

In [6]:
joined_df = pd.read_csv("../processed/joined_transit_data.csv")

/var/folders/vt/__g1vtqn7l13jfkknb257t7r0000gn/T/ipykernel_47510/2390423915.py:1: DtypeWarning: Columns (0,1,6,11,12,14,16,17,28,29,30,32,34,36) have mixed types. Specify dtype option on import or set low_memory=False.
  joined_df = pd.read_csv("../processed/joined_transit_data.csv")


In [7]:
joined_df = joined_df.dropna(subset=['route_id'])

In [8]:
joined_df['route_id'] = joined_df['route_id'].astype(str)
joined_df['trip_id'] = joined_df['trip_id'].astype(str)
joined_df['direction_id'] = joined_df['direction_id'].astype(str)

In [9]:
joined_df[joined_df['route_id']==100030][['stop_lat', 'stop_lon', 'trip_id', 'stop_sequence']]

,stop_lat,stop_lon,trip_id,stop_sequence


In [10]:
joined_df["n_stops"] = joined_df.groupby("trip_id")["stop_id"].transform("count")

In [11]:
longest_trips = (
    joined_df[["trip_id", "route_id", "direction_id", "n_stops"]]
    .drop_duplicates()
    .sort_values("n_stops", ascending=False)
    .drop_duplicates(["route_id", "direction_id"])
)

In [12]:
longest_trips[longest_trips['route_id']=='515']

,trip_id,route_id,direction_id,n_stops
675884,15046848__KPOB-CS:253:2:Thu-Fri:1:25SEP:85043:45,515,1.0,10
668370,13790001__KPOB-CS:221:1:Mon-Wed:1:25SEP:85011:123,515,0.0,9


In [13]:
filtered_df = joined_df[
    joined_df["trip_id"].isin(longest_trips["trip_id"])
]

In [14]:
filtered_df = filtered_df[['trip_id', 'stop_id', 'stop_sequence', 'route_id', 'shape_id', 'stop_lat', 'stop_lon', 'arrival_time', 'departure_time', 'direction_id']]
filtered_df = filtered_df.dropna(subset=['route_id', 'stop_sequence'])

In [15]:
filtered_df[filtered_df['route_id']==515]

,trip_id,stop_id,stop_sequence,route_id,shape_id,stop_lat,stop_lon,arrival_time,departure_time,direction_id


In [16]:
sorted_df = filtered_df.sort_values(['route_id', 'direction_id', 'stop_sequence'])

In [17]:
sorted_df["unique_key"] = "stop_" + sorted_df["stop_id"].astype(str) \
    + "_route_" + sorted_df["route_id"].astype(str) \
        + "_direction_" + sorted_df["direction_id"].astype(str)

In [18]:
sorted_df["next_stop"] = sorted_df.groupby(["route_id", "direction_id"])["unique_key"].shift(-1)

In [19]:
sorted_df[sorted_df['route_id']=='10030']

,trip_id,stop_id,stop_sequence,route_id,shape_id,stop_lat,stop_lon,arrival_time,departure_time,direction_id,unique_key,next_stop


In [20]:
def to_seconds(row):
    split = row.split(":")
    return int(split[0]) * 3600 + int(split[1]) * 60 + int(split[2])

sorted_df['arrival_time_seconds'] = sorted_df['arrival_time'].apply(to_seconds)
sorted_df['departure_time_seconds'] = sorted_df['departure_time'].apply(to_seconds)

In [21]:
sorted_df['estimated_travel_time_minutes_between_stops'] = (
    sorted_df.groupby(['route_id', 'direction_id'])['departure_time_seconds'].shift(-1) - sorted_df['arrival_time_seconds']
    ) / 60

In [22]:
sorted_df[sorted_df['route_id'].isna()]

,trip_id,stop_id,stop_sequence,route_id,shape_id,stop_lat,stop_lon,arrival_time,departure_time,direction_id,unique_key,next_stop,arrival_time_seconds,departure_time_seconds,estimated_travel_time_minutes_between_stops


In [23]:
cleaned_df = sorted_df.dropna(subset=["next_stop"])

In [24]:
cleaned_df.to_csv('transit.csv')

In [25]:
64080	- 63960	

120

In [26]:
cleaned_df[cleaned_df['route_id']==515]

,trip_id,stop_id,stop_sequence,route_id,shape_id,stop_lat,stop_lon,arrival_time,departure_time,direction_id,unique_key,next_stop,arrival_time_seconds,departure_time_seconds,estimated_travel_time_minutes_between_stops


In [27]:
cleaned_df[cleaned_df['estimated_travel_time_minutes_between_stops'] < 0]

,trip_id,stop_id,stop_sequence,route_id,shape_id,stop_lat,stop_lon,arrival_time,departure_time,direction_id,unique_key,next_stop,arrival_time_seconds,departure_time_seconds,estimated_travel_time_minutes_between_stops


In [28]:
cleaned_df[cleaned_df['route_id']=='100030']

,trip_id,stop_id,stop_sequence,route_id,shape_id,stop_lat,stop_lon,arrival_time,departure_time,direction_id,unique_key,next_stop,arrival_time_seconds,departure_time_seconds,estimated_travel_time_minutes_between_stops
203944,721370839,1-7430,1,100030,21131002,47.617504,-122.345268,07:38:00,07:38:00,0.0,stop_1-7430_route_100030_direction_0.0,stop_1-400_route_100030_direction_0.0,27480,27480,2.300000
203945,721370839,1-400,2,100030,21131002,47.614300,-122.344269,07:40:18,07:40:18,0.0,stop_1-400_route_100030_direction_0.0,stop_1-420_route_100030_direction_0.0,27618,27618,1.466667
203946,721370839,1-420,3,100030,21131002,47.612438,-122.341110,07:41:46,07:41:46,0.0,stop_1-420_route_100030_direction_0.0,stop_1-433_route_100030_direction_0.0,27706,27706,2.233333
203947,721370839,1-433,4,100030,21131002,47.609070,-122.337288,07:44:00,07:44:00,0.0,stop_1-433_route_100030_direction_0.0,stop_1-468_route_100030_direction_0.0,27840,27840,2.116667
203948,721370839,1-468,5,100030,21131002,47.606442,-122.334900,07:46:07,07:46:07,0.0,stop_1-468_route_100030_direction_0.0,stop_1-490_route_100030_direction_0.0,27967,27967,2.300000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
403536,786610029,1-531,46,100030,10131004,47.602642,-122.331184,24:54:57,24:54:57,1.0,stop_1-531_route_100030_direction_1.0,stop_1-548_route_100030_direction_1.0,89697,89697,1.933333
403537,786610029,1-548,47,100030,10131004,47.605541,-122.333832,24:56:53,24:56:53,1.0,stop_1-548_route_100030_direction_1.0,stop_1-570_route_100030_direction_1.0,89813,89813,2.116667
403538,786610029,1-570,48,100030,10131004,47.608688,-122.336700,24:59:00,24:59:00,1.0,stop_1-570_route_100030_direction_1.0,stop_1-590_route_100030_direction_1.0,89940,89940,1.316667
403539,786610029,1-590,49,100030,10131004,47.611137,-122.338951,25:00:19,25:00:19,1.0,stop_1-590_route_100030_direction_1.0,stop_1-600_route_100030_direction_1.0,90019,90019,1.216667


Generate distances between stops

In [29]:
coords = np.radians(cleaned_df[['stop_lat', 'stop_lon']])
distance_allowed_miles = .5
tree = BallTree(coords, metric='haversine')
radius = 0.25 / 3959  # 1 mile in radians

indices, dist = tree.query_radius(coords, r=radius, return_distance=True)
dist_miles = dist * 3959

In [30]:
rows = []

node_ids = cleaned_df['unique_key'].values
route_ids = cleaned_df['route_id'].values

for i, neighbors in enumerate(indices):
    current_id = node_ids[i]
    current_route = route_ids[i]

    for j, neighbor_idx in enumerate(neighbors):
        nearby_id = node_ids[neighbor_idx]
        nearby_route = route_ids[neighbor_idx]

        # skip self
        if current_id == nearby_id:
            continue

        # skip same-route stops
        if pd.notna(current_route) and pd.notna(nearby_route) and current_route == nearby_route:
            continue

        distance = dist_miles[i][j]

        # transfer time: 10 min
        transfer_time = 10
        estimated_time = (distance / 2) * 60 + transfer_time

        rows.append((current_id, nearby_id, estimated_time))

df_nearby_stops = pd.DataFrame(
    rows,
    columns=['source_node', 'nearby_node', 'estimated_time']
)

In [31]:
df_nearby_stops

,source_node,nearby_node,estimated_time
0,stop_21143_route_1_direction_0.0,stop_2238_route_2_direction_0.0,4.356107
1,stop_21143_route_1_direction_0.0,stop_164_route_2_direction_1.0,5.633604
2,stop_21143_route_1_direction_0.0,stop_164_route_52_direction_0.0,5.633604
3,stop_21143_route_1_direction_0.0,stop_1351_route_10_direction_0.0,3.371736
4,stop_21143_route_1_direction_0.0,stop_2238_route_52_direction_1.0,4.356107
...,...,...,...
252801,stop_T23-T1_route_TLINE_direction_1.0,stop_2869_route_45_direction_1.0,6.800738
252802,stop_T23-T1_route_TLINE_direction_1.0,stop_2143_route_28_direction_0.0,6.383749
252803,stop_T23-T1_route_TLINE_direction_1.0,stop_2136_route_28_direction_0.0,4.288350
252804,stop_T23-T1_route_TLINE_direction_1.0,stop_2159_route_28_direction_1.0,5.075191


In [32]:
df_nearby_stops['pair'] = df_nearby_stops.apply(
    lambda r: tuple(sorted((r['source_node'], r['nearby_node']))),
    axis=1
)

df_nearby_stops = df_nearby_stops.drop_duplicates('pair').drop(columns='pair')

In [33]:
df_nearby_stops

,source_node,nearby_node,estimated_time
0,stop_21143_route_1_direction_0.0,stop_2238_route_2_direction_0.0,4.356107
1,stop_21143_route_1_direction_0.0,stop_164_route_2_direction_1.0,5.633604
2,stop_21143_route_1_direction_0.0,stop_164_route_52_direction_0.0,5.633604
3,stop_21143_route_1_direction_0.0,stop_1351_route_10_direction_0.0,3.371736
4,stop_21143_route_1_direction_0.0,stop_2238_route_52_direction_1.0,4.356107
...,...,...,...
251758,stop_20-118_route_98_direction_1.0,stop_1039_route_99_direction_1.0,6.399332
251762,stop_20-118_route_98_direction_1.0,stop_1024_route_99_direction_0.0,6.670979
251836,stop_1017_route_99_direction_1.0,stop_1017_route_TA029_direction_1.0,0.000000
251878,stop_3046_route_STCL_direction_0.0,stop_T01_route_TLINE_direction_1.0,3.494082


In [34]:
df_nearby_stops.to_csv("transfers.csv")